### 🔎 Dilated Sliding Window Attention (Detailed Explanation)

Dilated Sliding Window Attention is an extension of **sliding window attention** that allows a model to **capture long-range dependencies while still keeping computation efficient**.

Instead of looking at **every token inside a window**, the model **skips tokens at fixed intervals** (called dilation).

This idea is inspired by **dilated convolutions used in CNNs**.

---

# 1️⃣ Problem With Normal Sliding Window

In **sliding window attention**, each token attends only to nearby tokens.

Allowed tokens:

$$
j \in [i-W+1, i]
$$

Where:

* (i) → current token
* (W) → window size

Example:

```
Tokens:
T0 T1 T2 T3 T4 T5 T6 T7
```

Window = **4**

Attention pattern:

```
T4 → T1 T2 T3 T4
```

So token **T4** can only see **4 nearby tokens**.

---

⚠️ Problem:

Even if the sequence is very long:

```
Sequence = 16K tokens
Window = 512
```

The model **cannot directly see far tokens**.

Long-range information must travel through **many layers**, which is inefficient.

---

# 2️⃣ Idea of Dilated Sliding Window

Dilated attention **skips tokens with a step size (dilation rate)**.

Instead of attending to **every token in the window**, it attends to **spaced tokens**.

General rule:

$$
j = i - k \times d
$$

Where:

* (d) → dilation rate
* (k) → index inside window

---

# 3️⃣ Example

Sequence:

```
T0 T1 T2 T3 T4 T5 T6 T7 T8 T9
```

Current token = **T9**

Window size = **4**

---

### Normal Sliding Window

```
T9 → T6 T7 T8 T9
```

Only nearby tokens.

---

### Dilated Sliding Window (dilation = 2)

```
T9 → T3 T5 T7 T9
```

The model now sees **farther tokens**.

---

### Dilated Sliding Window (dilation = 3)

```
T9 → T0 T3 T6 T9
```

Even larger coverage.

---

# 4️⃣ Visual Representation

Normal sliding window:

```
T0 T1 T2 T3 T4 T5 T6 T7 T8 T9
               ↑
         [T6 T7 T8 T9]
```

Dilated window (d=2):

```
T0 T1 T2 T3 T4 T5 T6 T7 T8 T9
   ↑   ↑   ↑   ↑
  T3  T5  T7  T9
```

Dilated window (d=3):

```
T0 T1 T2 T3 T4 T5 T6 T7 T8 T9
↑     ↑     ↑     ↑
T0    T3    T6    T9
```

---

# 5️⃣ Attention Formula

The attention computation stays the same:

$$
Attention(Q,K,V) =
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

But the **mask selects dilated tokens**.

Allowed indices:

$$
j = i - k \times d
$$

---

# 6️⃣ Complexity

Normal attention:

$$
O(N^2)
$$

Sliding window:

$$
O(NW)
$$

Dilated sliding window:

$$
O(NW)
$$

But **effective receptive field becomes much larger**.

---

# 7️⃣ Receptive Field Improvement

Example:

```
Window = 8
```

Normal sliding window coverage:

```
8 tokens
```

Dilated sliding window (d=4):

```
8 × 4 = 32 tokens
```

So the model sees **4× larger context** with same compute.

---

# 8️⃣ Why It Is Useful

Dilated sliding window solves a key issue:

```
local attention  → limited context
global attention → expensive
```

Dilated attention provides a **middle ground**.

Benefits:

✅ larger context coverage
✅ same computational cost
✅ efficient for long sequences
✅ captures long-range dependencies

---

# 9️⃣ Where It Is Used

Dilated attention appears in long-context transformer architectures such as:

* Longformer
* BigBird
* Sparse Transformer

These models combine:

* sliding window attention
* dilated attention
* global attention tokens

---

# 🔟 Comparison

| Method                 | Tokens Seen   | Complexity |
| ---------------------- | ------------- | ---------- |
| Full Attention         | all tokens    | (O(N^2))   |
| Sliding Window         | nearby tokens | (O(NW))    |
| Dilated Sliding Window | spaced tokens | (O(NW))    |

---

# 1️⃣1️⃣ Intuition

Imagine reading a **long document**.

Normal sliding window:

```
You only read nearby sentences
```

Dilated sliding window:

```
You read nearby sentences
+ occasionally jump far back
```

This helps the model remember **important earlier context**.

---

# 1️⃣2️⃣ Practical Usage

In real LLM implementations:

Dilated attention is often combined with:

* **sliding window attention**
* **global tokens**
* **flash attention kernels**

This gives:

```
Local context + Long context + Efficient computation
```

---

✅ **Summary**

Dilated Sliding Window Attention:

* skips tokens using a dilation step
* increases effective context coverage
* keeps computation (O(NW))
* helps models capture long-range dependencies.

---

If you want, I can also show you:

* **PyTorch implementation of Dilated Sliding Window Attention**
* **how it differs from Ring Attention**
* **how Mistral-style sliding attention works internally**

Those differences are **very important for modern LLM architectures.** 🚀


In [1]:
import torch
import torch.nn as nn
import math


class DilatedSlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, window_size, dilation):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.dilation = dilation
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def create_dilated_mask(self, seq_len, device):

        mask = torch.full((seq_len, seq_len), float('-inf'), device=device)

        for i in range(seq_len):

            for k in range(self.window_size):

                j = i - k * self.dilation

                if j >= 0:
                    mask[i, j] = 0

        return mask

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1))

        scores = scores / math.sqrt(self.head_dim)

        mask = self.create_dilated_mask(T, x.device)

        scores = scores + mask

        attn = torch.softmax(scores, dim=-1)

        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(out)